# 10 CV Baseline

Run this notebook after `00_data_loading_and_split.ipynb`. It evaluates a restart-safe constant-velocity trajectory baseline and saves metrics, plots, and predictions to disk.


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "model_training"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from training.notebook_workflow import (
    describe_shared_artifacts,
    finish_runtime_report,
    load_or_build_shared_artifacts,
    prepare_result_dirs,
    run_constant_velocity_baseline,
    start_runtime_report,
    set_seed,
    timestamp_tag,
)


In [2]:
SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15


In [3]:
shared_runtime = start_runtime_report(
    stage_name="shared_data_split",
    output_dir=PIPELINE_ROOT / "results",
    context={"past_len": PAST_LEN, "future_len": FUTURE_LEN, "seed": SEED},
)
set_seed(SEED)
streams, sample_table, split_info, sample_table_path, split_path = load_or_build_shared_artifacts(
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
)
print("Sample table:", sample_table_path)
print("Split path:", split_path)
print("Split strategy:", split_info["split_strategy"])
print("Episode count:", split_info["episode_count"])
print("Train / Val / Test samples:", len(split_info["train_indices"]), len(split_info["val_indices"]), len(split_info["test_indices"]))
print("Train episodes:", split_info["train_episode_ids"])
print("Val episodes:", split_info["val_episode_ids"])
print("Test episodes:", split_info["test_episode_ids"])
shared_runtime_summary = finish_runtime_report(
    shared_runtime,
    extra=describe_shared_artifacts(
        streams=streams,
        sample_table=sample_table,
        split_info=split_info,
        sample_table_path=sample_table_path,
        split_path=split_path,
    ),
)
print(json.dumps(shared_runtime_summary, indent=2))


[runtime] shared_data_split start: 2026-05-23T11:22:40 summary=/home/basudeo/Documents/Thesis/model_training/results/runtime/20260523_112240_shared_data_split_runtime_summary.json
Sample table: /home/basudeo/Documents/Thesis/model_training/results/train_ready/sample_table_seed42_past10_future5.json
Split path: /home/basudeo/Documents/Thesis/model_training/results/splits/trajectory_split_seed42_past10_future5.json
Split strategy: episode
Episode count: 8
Train / Val / Test samples: 36405 6082 9219
Train episodes: ['run_model_20260522_203433', 'run_model_20260522_204215', 'run_model_20260523_013946', 'run_model_20260523_015017', 'run_model_20260522_202537']
Val episodes: ['run_model_20260523_013116']
Test episodes: ['run_model_20260522_201207', 'run_model_20260522_201854']
[runtime] shared_data_split done: start=2026-05-23T11:22:40 end=2026-05-23T11:22:44 elapsed_s=4.348 peak_cpu=10.70% peak_rss_mb=1046.01 peak_gpu_alloc_mb=0.00
{
  "stage_name": "shared_data_split",
  "timestamp": "2026

In [4]:
MODEL_SLUG = "cv_baseline"
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)
cv_runtime = start_runtime_report(
    stage_name=f"{MODEL_SLUG}_notebook_run",
    output_dir=result_dir,
    context={
        "model_slug": MODEL_SLUG,
        "timestamp": TIMESTAMP,
        "past_len": PAST_LEN,
        "future_len": FUTURE_LEN,
        "shared_runtime_summary": shared_runtime_summary,
    },
)
final_metrics = run_constant_velocity_baseline(
    streams=streams,
    sample_table=sample_table,
    split_info=split_info,
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    plot_dir=plot_dir,
    runtime_output_dir=result_dir,
    runtime_context=describe_shared_artifacts(
        streams=streams,
        sample_table=sample_table,
        split_info=split_info,
        sample_table_path=sample_table_path,
        split_path=split_path,
    ),
)
final_metrics["notebook_runtime"] = finish_runtime_report(
    cv_runtime,
    extra={
        **describe_shared_artifacts(
            streams=streams,
            sample_table=sample_table,
            split_info=split_info,
            sample_table_path=sample_table_path,
            split_path=split_path,
        ),
        "model_slug": MODEL_SLUG,
    },
)
print(json.dumps(final_metrics, indent=2))


[runtime] cv_baseline_notebook_run start: 2026-05-23T11:22:44 summary=/home/basudeo/Documents/Thesis/model_training/results/cv_baseline/runtime/20260523_112244_cv_baseline_notebook_run_runtime_summary.json
[runtime] cv_baseline_baseline start: 2026-05-23T11:22:44 summary=/home/basudeo/Documents/Thesis/model_training/results/cv_baseline/runtime/20260523_112244_cv_baseline_baseline_runtime_summary.json
[runtime] cv_baseline_baseline done: start=2026-05-23T11:22:44 end=2026-05-23T11:22:46 elapsed_s=2.096 peak_cpu=25.80% peak_rss_mb=1013.59 peak_gpu_alloc_mb=0.00
[runtime] cv_baseline_notebook_run done: start=2026-05-23T11:22:44 end=2026-05-23T11:22:47 elapsed_s=3.005 peak_cpu=25.50% peak_rss_mb=1021.74 peak_gpu_alloc_mb=0.00
{
  "ADE": 0.09883134812116623,
  "FDE": 0.13863298296928406,
  "RMSE": 0.16410906612873077,
  "loss": 0.09883134812116623,
  "model_slug": "cv_baseline",
  "timestamp": "20260523_112244",
  "split_path": "/home/basudeo/Documents/Thesis/model_training/results/splits/t